test

In [11]:
import pandas as pd
import numpy as np

import tensorflow as tf
from tensorflow.keras import Sequential, Input
from tensorflow.keras.layers import Dense, Dropout
from tensorflow.keras.callbacks import EarlyStopping

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score
)

import matplotlib.pyplot as plt
import seaborn as sns

print("TensorFlow Version:", tf.__version__)

TensorFlow Version: 2.21.0


In [12]:
df = pd.read_csv("../data/mpg_clean.csv")

df.head()

,mpg,cylinders,displacement,horsepower,weight,acceleration,model_year,origin_japan,origin_usa
0,18.0,8,307.0,130.0,3504,12.0,70,0,1
1,15.0,8,350.0,165.0,3693,11.5,70,0,1
2,18.0,8,318.0,150.0,3436,11.0,70,0,1
3,16.0,8,304.0,150.0,3433,12.0,70,0,1
4,17.0,8,302.0,140.0,3449,10.5,70,0,1


In [13]:
print("Shape:", df.shape)

df.info()

Shape: (392, 9)
<class 'pandas.DataFrame'>
RangeIndex: 392 entries, 0 to 391
Data columns (total 9 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   mpg           392 non-null    float64
 1   cylinders     392 non-null    int64  
 2   displacement  392 non-null    float64
 3   horsepower    392 non-null    float64
 4   weight        392 non-null    int64  
 5   acceleration  392 non-null    float64
 6   model_year    392 non-null    int64  
 7   origin_japan  392 non-null    int64  
 8   origin_usa    392 non-null    int64  
dtypes: float64(4), int64(5)
memory usage: 27.7 KB


## Features and Target

In [14]:
X = df.drop("mpg", axis=1)

y = df["mpg"]

print("Features Shape:", X.shape)
print("Target Shape:", y.shape)

Features Shape: (392, 8)
Target Shape: (392,)


## Train-Test Split

In [15]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)

print("Training Samples:", len(X_train))
print("Testing Samples:", len(X_test))

Training Samples: 313
Testing Samples: 79


## Feature Scaling

In [16]:
# Neural Networks perform much better when features are scaled.

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

## Build Neural Network

In [18]:
model = Sequential([
    
    Input(shape=(X_train_scaled.shape[1],)),  # Input layer first

    Dense(
        64,                         # 64 neurons (brain cells)
        activation="relu",          # Activation function (how neurons fire)
    ),

    Dropout(0.2),    # Randomly turn off 20% of neurons during training and it Prevents overfitting

    Dense(
        32,                           # 32 neurons (fewer than first layer)
        activation="relu"             # Same activation function
    ),

    Dropout(0.2),                   # Another 20% dropout for regularization

    Dense(1)                        # Single neuron, no activation (linear output)
])

## Neural Network Architecture - Quick Reference

### What It Does
Builds a neural network "brain" to predict MPG from car features

### Data Flow


### Layer Explanations

#### 1. `Dense(64, activation="relu", input_shape=(8,))`
- **64 neurons** = 64 "detectors" looking for patterns
- **ReLU** = neuron fires only if pattern found (ignores negatives)
- **input_shape** = receives 8 car features

#### 2. `Dropout(0.2)`
- Randomly turns off 20% of neurons during training
- Prevents overfitting (memorizing instead of learning)

#### 3. `Dense(32, activation="relu")`
- 32 neurons combine patterns from previous layer
- Fewer neurons = compressing information

#### 4. `Dropout(0.2)`
- Another 20% dropout for regularization

#### 5. `Dense(1)`
- Single neuron produces final MPG prediction
- No activation = can output any number (like 25.7)

### Why These Numbers?

| Parameter | Reason |
|-----------|--------|
| 64 neurons | Good for ~400 samples (not too big/small) |
| 32 neurons | Typically half of previous layer |
| 0.2 dropout | Standard starting point (10-30% common) |
| 2 hidden layers | Enough for tabular data |

### Memory Aid
> "8 inputs → think broadly (64) → drop some (0.2) → focus (32) → drop again (0.2) → output (1)"

> "Wider first layer, narrower second layer, dropout in between"

### Common Patterns

| Dataset Size | Architecture |
|--------------|--------------|
| Small (<1000 samples) | `[32, 16]` |
| Medium (<10000 samples) | `[64, 32] + Dropout 0.2` ← **YOUR CASE** |
| Large (>10000 samples) | `[128, 64, 32] + Dropout 0.3` |

### Alternatives

#### Simpler (if overfitting)
```python
Dense(32, activation='relu')
Dense(16, activation='relu')
Dense(1)

## Model Summary

In [19]:
model.summary()

Model: "sequential_2"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━┓
┃ Layer (type)                         ┃ Output Shape                ┃         Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━┩
│ dense_7 (Dense)                      │ (None, 64)                  │             576 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dropout_4 (Dropout)                  │ (None, 64)                  │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dense_8 (Dense)                      │ (None, 32)                  │           2,080 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dropout_5 (Dropout)                  │ (None, 32)                  │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dense_9 (Dense)                      │ (None, 1)                   │              33 │
└──────────────────────────────────────┴─────────────────────────────┴─────────────────┘

 Total params: 2,689 (10.50 KB)

 Trainable params: 2,689 (10.50 KB)

 Non-trainable params: 0 (0.00 B)

## Compile Model

In [20]:
model.compile(
    optimizer="adam",
    loss="mae",
    metrics=["mae","mse"]
)

## Early Stopping

In [22]:
early_stopping = EarlyStopping(
    monitor="val_loss",        # What to watch (validation loss)
    patience=20,               # How many epochs to wait for improvement
    restore_best_weights=True  # Keep the best version, not the last
)

## Train Model

In [23]:
history = model.fit(
    X_train_scaled,      # Training features (scaled)
    y_train,             # Training targets (actual MPG values)
    validation_split=0.2, # Use 20% of training data for validation
    epochs=200,          # Maximum training rounds
    batch_size=32,       # How many cars to learn from at once
    callbacks=[early_stopping], # Smart stopping
    verbose=1            # Show progress bars
)

Epoch 1/200
8/8 ━━━━━━━━━━━━━━━━━━━━ 1s 31ms/step - loss: 23.0773 - mae: 23.0773 - mse: 596.5793 - val_loss: 24.6005 - val_mae: 24.6005 - val_mse: 667.5279
Epoch 2/200
8/8 ━━━━━━━━━━━━━━━━━━━━ 0s 10ms/step - loss: 22.5848 - mae: 22.5848 - mse: 575.6069 - val_loss: 24.1337 - val_mae: 24.1337 - val_mse: 645.7885
Epoch 3/200
8/8 ━━━━━━━━━━━━━━━━━━━━ 0s 10ms/step - loss: 22.0519 - mae: 22.0519 - mse: 554.1135 - val_loss: 23.5972 - val_mae: 23.5972 - val_mse: 621.8253
Epoch 4/200
8/8 ━━━━━━━━━━━━━━━━━━━━ 0s 10ms/step - loss: 21.4406 - mae: 21.4406 - mse: 530.3037 - val_loss: 22.9430 - val_mae: 22.9430 - val_mse: 593.6684
Epoch 5/200
8/8 ━━━━━━━━━━━━━━━━━━━━ 0s 10ms/step - loss: 20.6234 - mae: 20.6234 - mse: 500.1183 - val_loss: 22.1368 - val_mae: 22.1368 - val_mse: 560.9342
Epoch 6/200
8/8 ━━━━━━━━━━━━━━━━━━━━ 0s 9ms/step - loss: 19.6012 - mae: 19.6012 - mse: 464.6854 - val_loss: 21.0991 - val_mae: 21.0991 - val_mse: 521.7468
Epoch 7/200
8/8 ━━━━━━━━━━━━━━━━━━━━ 0s 10ms/step - loss: 18.3709

## Training History

In [24]:
history_df = pd.DataFrame(history.history)

history_df.head()

,loss,mae,mse,val_loss,val_mae,val_mse
0,23.077349,23.077349,596.579346,24.600458,24.600458,667.527893
1,22.584801,22.584801,575.606934,24.133652,24.133652,645.788452
2,22.051908,22.051908,554.113525,23.597218,23.597218,621.825256
3,21.440563,21.440563,530.303711,22.943035,22.943035,593.668396
4,20.623367,20.623367,500.118317,22.136801,22.136801,560.934204
